In [1]:
# Import required libraries
import pandas as pd
import numpy as np

# Load the diabetes hospital readmission dataset
df = pd.read_csv('/content/diabetic_data.csv')

# Display the first five rows
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [2]:
# Check the size of the dataset
print(df.shape)
print("Number of rows:",df.shape[0])
print("Number of columns:",df.shape[1])
# Check column names, data types and missing values
df.info()

(101766, 50)
Number of rows: 101766
Number of columns: 50
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  n

In [3]:
# Check for duplicate rows
print("Number of duplicate rows:",df.duplicated().sum())
# Check missing values
df.isnull().sum()

Number of duplicate rows: 0


,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0


In [4]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Check missing values again
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,2273
gender,0
age,0
weight,98569
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0


In [5]:
# Create binary target for 30-day readmission
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

# Check the target distribution
print(df['readmitted_30'].value_counts())

readmitted_30
0    90409
1    11357
Name: count, dtype: int64


In [6]:
# Remove ID columns and the original target column
df.drop(['encounter_id', 'patient_nbr', 'readmitted'], axis=1, inplace=True)
# Check the remaining columns
print(df.shape)
print(df.columns)

(101766, 48)
Index(['race', 'gender', 'age', 'weight', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'payer_code', 'medical_specialty', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin',
       'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
       'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted_30'],
      dtype='object')


In [7]:
# Select features relevant to 30-day readmission prediction
features = [
    'age',
    'gender',
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
    'diag_1',
    'diag_2',
    'diag_3',
    'max_glu_serum',
    'A1Cresult'
]

In [8]:
# Create feature dataset
X = df[features]

# Create target variable
y = df['readmitted_30']

# Display the selected features
X.head()

,age,gender,time_in_hospital,num_lab_procedures,num_procedures,number_outpatient,number_emergency,number_inpatient,number_diagnoses,diag_1,diag_2,diag_3,max_glu_serum,A1Cresult
0,[0-10),Female,1,41,0,0,0,0,1,250.83,NaN,NaN,NaN,NaN
1,[10-20),Female,3,59,0,0,0,0,9,276,250.01,255,NaN,NaN
2,[20-30),Female,2,11,5,2,0,1,6,648,250,V27,NaN,NaN
3,[30-40),Male,2,44,1,0,0,0,7,8,250.43,403,NaN,NaN
4,[40-50),Male,1,51,0,0,0,0,5,197,157,250,NaN,NaN


In [9]:
# Make a copy of selected features
X = X.copy()

# Fill missing values in categorical columns with 'Unknown'
categorical_columns = [
    'age',
    'gender',
    'diag_1',
    'diag_2',
    'diag_3',
    'max_glu_serum',
    'A1Cresult'
]

X[categorical_columns] = X[categorical_columns].fillna('Unknown')

# Check remaining missing values
print(X.isnull().sum())

age                   0
gender                0
time_in_hospital      0
num_lab_procedures    0
num_procedures        0
number_outpatient     0
number_emergency      0
number_inpatient      0
number_diagnoses      0
diag_1                0
diag_2                0
diag_3                0
max_glu_serum         0
A1Cresult             0
dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [13]:
# Convert categorical columns into numerical columns
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Make sure training and testing have the same columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Convert True/False values to 0/1
X_train = X_train.astype(int)
X_test = X_test.astype(int)

# Check the new shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (81412, 2191)
X_test: (20354, 2191)


In [15]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Scale the training and testing features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
from sklearn.linear_model import LogisticRegression

# Create Logistic Regression model with L2 regularization
model = LogisticRegression(penalty='l2',C=1.0,max_iter=1000,random_state=42)

# Train the model
model.fit(X_train_scaled, y_train)

print("Model training completed.")

Model training completed.


In [17]:
# Predict the class labels
y_pred = model.predict(X_test_scaled)

# Predict probability of 30-day readmission
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("First 20 predictions:")
print(y_pred[:20])

print("\nFirst 20 readmission probabilities:")
print(y_prob[:20])

First 20 predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

First 20 readmission probabilities:
[0.13572572 0.05875925 0.0915436  0.20539693 0.07070447 0.0394212
 0.06383951 0.07698623 0.18791374 0.18894094 0.06742094 0.14020327
 0.07086339 0.05922351 0.06273602 0.14373216 0.07441579 0.06989561
 0.05203354 0.11912207]


In [18]:
from sklearn.metrics import roc_auc_score

# Calculate ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC Score:", roc_auc)

ROC-AUC Score: 0.6311484645158281


In [19]:
from sklearn.metrics import confusion_matrix

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[18001    82]
 [ 2213    58]]


In [20]:
# Create a DataFrame containing actual and predicted values
results = pd.DataFrame({
    'Actual_Readmission': y_test.values,
    'Predicted_Readmission': y_pred,
    'Readmission_Probability': y_prob
})

# Save the predictions
results.to_csv('hospital_readmission_predictions.csv', index=False)

# Display the first five results
results.head()

,Actual_Readmission,Predicted_Readmission,Readmission_Probability
0,0,0,0.135726
1,0,0,0.058759
2,0,0,0.091544
3,1,0,0.205397
4,1,0,0.070704
